# Chapter 9 — Parallel histogram

Based on chapter 9 of *Programming Massively Parallel Processors* (Hwu, Kirk, El Hajj). Every program in this folder is a single `.cu` file that runs on an NVIDIA GPU (Google Colab) **and** on any laptop without a GPU, through the CPU emulation in [`../common/cuda_compat.h`](../common/cuda_compat.h).

This notebook goes **step by step**: 7 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/07_histogram/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 7: `01_histogram_sequential.cu`

The sequential loop `histogram_sequential` on the tree image; it is called row by row so the bins are printed after every row. (slide: *What you already know*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_histogram_sequential.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

const unsigned int EXPECTED_BINS[NUM_BINS] = {6, 12, 14, 32};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBinHeader() {
  std::printf("%-12s", "bin");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" [%3u-%3u]", b * INTENSITIES_PER_BIN, (b + 1) * INTENSITIES_PER_BIN - 1);
  }
  std::printf("\n");
}

static void printBins(const char* label, const unsigned int* bins) {
  std::printf("%-12s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
  }
  std::printf("\n");
}

static bool sameBins(const unsigned int* bins, const unsigned int* expected) {
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

int main() {
  printImage();
  std::printf("\nbin = intensity / %u; the loop visits the pixels row by row\n",
              INTENSITIES_PER_BIN);
  printBinHeader();

  unsigned int bins[NUM_BINS] = {0, 0, 0, 0};
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    histogram_sequential(TREE_IMAGE[row], bins, WIDTH, 1);
    char label[16];
    std::snprintf(label, sizeof(label), "after row %u", row);
    printBins(label, bins);
  }
  printBins("expected", EXPECTED_BINS);

  const bool ok = sameBins(bins, EXPECTED_BINS);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_histogram_sequential.cu -o 01_histogram_sequential && ./01_histogram_sequential

**What to look at in the output:** The bins grow row by row: rows 0 to 3 add only to the leaves and sky bins, and the last line equals the expected 6, 12, 14, 32.

## Step 2 of 7: `02_race_and_atomic.cu`

One thread per pixel. `histogram_racy` uses a plain `++bins[b]` and loses updates on a GPU (the CPU emulation often loses none); `histogram_racy_all_read_first` puts a barrier between the read and the write, so every thread reads before anyone writes and each bin ends at 1 (60 updates lost). `histogram_basic` uses `cuda::atomic_ref` and `fetch_add` and is exact. The two racy kernels are wrong on purpose: ThreadSanitizer reports them, and the PASS refers to the atomic kernel. (slides: *Atomic operations in CUDA C++*, *Code: the basic histogram kernel*, *Why `memory_order_relaxed` is enough*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_race_and_atomic.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_PIXELS = WIDTH * HEIGHT;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;
constexpr unsigned int THREADS_PER_BLOCK = 64;

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_racy(const unsigned char* image,
                               unsigned int* bins,
                               unsigned int width,
                               unsigned int height) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_racy_all_read_first(const unsigned char* image,
                                              unsigned int* bins,
                                              unsigned int width,
                                              unsigned int height) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  bool hasPixel = i < width * height;
  unsigned char b = hasPixel ? image[i] / INTENSITIES_PER_BIN : 0;

  unsigned int oldValue = hasPixel ? bins[b] : 0;
  __syncthreads();

  if (hasPixel) {
    bins[b] = oldValue + 1;
  }
}

__global__ void histogram_basic(const unsigned char* image,
                                unsigned int* bins,
                                unsigned int width,
                                unsigned int height) {
  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
    bins_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBins(const char* label, const unsigned int* bins) {
  unsigned int counted = 0;
  std::printf("%-22s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
    counted += bins[b];
  }
  std::printf(" %9u\n", NUM_PIXELS - counted);
}

template <class Kernel>
static void runKernel(Kernel kernel, const unsigned char* image_d, unsigned int* bins) {
  unsigned int* bins_d = nullptr;
  CHECK(cudaMalloc(&bins_d, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMemset(bins_d, 0, NUM_BINS * sizeof(unsigned int)));

  launch(kernel,
         dim3(cdiv(NUM_PIXELS, THREADS_PER_BLOCK)),
         dim3(THREADS_PER_BLOCK),
         image_d,
         bins_d,
         WIDTH,
         HEIGHT);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(bins, bins_d, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(bins_d));
}

int main() {
  unsigned char* image_d = nullptr;
  CHECK(cudaMalloc(&image_d, NUM_PIXELS));
  CHECK(cudaMemcpy(image_d, &TREE_IMAGE[0][0], NUM_PIXELS, cudaMemcpyHostToDevice));

  unsigned int expected[NUM_BINS] = {0, 0, 0, 0};
  unsigned int racy[NUM_BINS];
  unsigned int allReadFirst[NUM_BINS];
  unsigned int atomic[NUM_BINS];
  histogram_sequential(&TREE_IMAGE[0][0], expected, WIDTH, HEIGHT);
  runKernel(histogram_racy, image_d, racy);
  runKernel(histogram_racy_all_read_first, image_d, allReadFirst);
  runKernel(histogram_basic, image_d, atomic);
  CHECK(cudaFree(image_d));

  printImage();
  std::printf("\n%u threads, one per pixel\n", NUM_PIXELS);
  std::printf("%-22s %9s %9s %9s %9s %9s\n",
              "kernel",
              "[0-63]",
              "[64-127]",
              "[128-191]",
              "[192-255]",
              "lost");
  printBins("sequential", expected);
  printBins("++bins[b]", racy);
  printBins("all read, then write", allReadFirst);
  printBins("atomic_ref fetch_add", atomic);

  bool ok = true;
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    ok = ok && atomic[b] == expected[b];
  }
  std::printf("%s\n", ok ? "PASS (atomic kernel exact)" : "FAIL (atomic kernel wrong)");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_race_and_atomic.cu -o 02_race_and_atomic && ./02_race_and_atomic

**What to look at in the output:** The `lost` column. The `++bins[b]` row may lose nothing on the CPU emulation but loses updates on a GPU; the "all read, then write" row has 1 in every bin (60 lost) on both; the atomic row equals the sequential one.

## Step 3 of 7: `03_private_global.cu`

`histogram_private_global`: 4 blocks of 16 threads, each block counts into its own slice of `bins_pool` (block scope), then commits it to `bins` (device scope). Every block's private copy is printed next to the total. (slide: *Code: private copies in global memory*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_private_global.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_PIXELS = WIDTH * HEIGHT;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;
constexpr unsigned int THREADS_PER_BLOCK = 16;
constexpr unsigned int NUM_BLOCKS = NUM_PIXELS / THREADS_PER_BLOCK;
constexpr unsigned int POOL_SIZE = NUM_BLOCKS * NUM_BINS;

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_private_global(const unsigned char* image,
                                         unsigned int* bins,
                                         unsigned int* bins_pool,
                                         unsigned int width,
                                         unsigned int height) {
  unsigned int* bins_priv = &bins_pool[blockIdx.x * NUM_BINS];

  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_priv_ref(bins_priv[b]);
    bins_priv_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_priv[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_priv[b], cuda::memory_order_relaxed);
    }
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBinHeader() {
  std::printf("%-12s", "bin");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" [%3u-%3u]", b * INTENSITIES_PER_BIN, (b + 1) * INTENSITIES_PER_BIN - 1);
  }
  std::printf("\n");
}

static void printBins(const char* label, const unsigned int* bins) {
  std::printf("%-12s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
  }
  std::printf("\n");
}

static bool sameBins(const unsigned int* bins, const unsigned int* expected) {
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

int main() {
  unsigned char* image_d = nullptr;
  unsigned int* bins_d = nullptr;
  unsigned int* bins_pool_d = nullptr;
  CHECK(cudaMalloc(&image_d, NUM_PIXELS));
  CHECK(cudaMalloc(&bins_d, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMalloc(&bins_pool_d, POOL_SIZE * sizeof(unsigned int)));
  CHECK(cudaMemcpy(image_d, &TREE_IMAGE[0][0], NUM_PIXELS, cudaMemcpyHostToDevice));
  CHECK(cudaMemset(bins_d, 0, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMemset(bins_pool_d, 0, POOL_SIZE * sizeof(unsigned int)));

  launch(histogram_private_global,
         dim3(NUM_BLOCKS),
         dim3(THREADS_PER_BLOCK),
         image_d,
         bins_d,
         bins_pool_d,
         WIDTH,
         HEIGHT);
  CHECK(cudaGetLastError());

  unsigned int bins[NUM_BINS];
  unsigned int bins_pool[POOL_SIZE];
  CHECK(cudaMemcpy(bins, bins_d, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(
      cudaMemcpy(bins_pool, bins_pool_d, POOL_SIZE * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(image_d));
  CHECK(cudaFree(bins_d));
  CHECK(cudaFree(bins_pool_d));

  unsigned int expected[NUM_BINS] = {0, 0, 0, 0};
  histogram_sequential(&TREE_IMAGE[0][0], expected, WIDTH, HEIGHT);

  printImage();
  std::printf(
      "\n%u blocks of %u threads: block k counts rows 2k and 2k + 1 in its slice of bins_pool\n",
      NUM_BLOCKS,
      THREADS_PER_BLOCK);
  printBinHeader();
  for (unsigned int block = 0; block < NUM_BLOCKS; ++block) {
    char label[16];
    std::snprintf(label, sizeof(label), "block %u", block);
    printBins(label, &bins_pool[block * NUM_BINS]);
  }
  printBins("bins (sum)", bins);
  printBins("sequential", expected);

  const bool ok = sameBins(bins, expected);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_private_global.cu -o 03_private_global && ./03_private_global

**What to look at in the output:** Each private copy counts only its block's two rows (block 0 sees only sky and leaves), and the four copies add up to the final bins.

## Step 4 of 7: `04_private_shared.cu`

`histogram_private_shared`: the private copy moves to `__shared__` memory; the block zeroes it itself, and two barriers separate zeroing, counting and committing. (slide: *Code: private copies in shared memory*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_private_shared.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_PIXELS = WIDTH * HEIGHT;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;
constexpr unsigned int THREADS_PER_BLOCK = 16;

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_private_shared(const unsigned char* image,
                                         unsigned int* bins,
                                         unsigned int width,
                                         unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < width * height) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
    bins_s_ref.fetch_add(1, cuda::memory_order_relaxed);
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBinHeader() {
  std::printf("%-12s", "bin");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" [%3u-%3u]", b * INTENSITIES_PER_BIN, (b + 1) * INTENSITIES_PER_BIN - 1);
  }
  std::printf("\n");
}

static void printBins(const char* label, const unsigned int* bins) {
  std::printf("%-12s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
  }
  std::printf("\n");
}

static bool sameBins(const unsigned int* bins, const unsigned int* expected) {
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

int main() {
  const unsigned int numBlocks = cdiv(NUM_PIXELS, THREADS_PER_BLOCK);

  unsigned char* image_d = nullptr;
  unsigned int* bins_d = nullptr;
  CHECK(cudaMalloc(&image_d, NUM_PIXELS));
  CHECK(cudaMalloc(&bins_d, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMemcpy(image_d, &TREE_IMAGE[0][0], NUM_PIXELS, cudaMemcpyHostToDevice));
  CHECK(cudaMemset(bins_d, 0, NUM_BINS * sizeof(unsigned int)));

  launch(histogram_private_shared,
         dim3(numBlocks),
         dim3(THREADS_PER_BLOCK),
         image_d,
         bins_d,
         WIDTH,
         HEIGHT);
  CHECK(cudaGetLastError());

  unsigned int bins[NUM_BINS];
  CHECK(cudaMemcpy(bins, bins_d, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(image_d));
  CHECK(cudaFree(bins_d));

  unsigned int expected[NUM_BINS] = {0, 0, 0, 0};
  histogram_sequential(&TREE_IMAGE[0][0], expected, WIDTH, HEIGHT);

  printImage();
  std::printf("\n%u blocks of %u threads, each with its copy bins_s in shared memory\n",
              numBlocks,
              THREADS_PER_BLOCK);
  printBinHeader();
  printBins("bins", bins);
  printBins("sequential", expected);

  const bool ok = sameBins(bins, expected);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_private_shared.cu -o 04_private_shared && ./04_private_shared

**What to look at in the output:** The same bins as in step 3, now without a pool of private copies in global memory.

## Step 5 of 7: `05_coarsening.cu`

`histogram_coarse_contig` and `histogram_coarse_interl`: 8 threads per block with `COARSE_FACTOR` = 4 pixels each, one line apart. The program prints the pixels block 0 reads in each iteration: 4 apart in the contiguous kernel, consecutive (coalesced) in the interleaved one. (slide: *Code: coarsening, one line apart*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_coarsening.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_PIXELS = WIDTH * HEIGHT;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;
constexpr unsigned int THREADS_PER_BLOCK = 8;
constexpr unsigned int COARSE_FACTOR = 4;

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_coarse_contig(const unsigned char* image,
                                        unsigned int* bins,
                                        unsigned int width,
                                        unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int i = segment + threadIdx.x * COARSE_FACTOR + c;
    if (i < width * height) {
      unsigned char b = image[i] / INTENSITIES_PER_BIN;
      cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
      bins_s_ref.fetch_add(1, cuda::memory_order_relaxed);
    }
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

__global__ void histogram_coarse_interl(const unsigned char* image,
                                        unsigned int* bins,
                                        unsigned int width,
                                        unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int i = segment + c * blockDim.x + threadIdx.x;
    if (i < width * height) {
      unsigned char b = image[i] / INTENSITIES_PER_BIN;
      cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
      bins_s_ref.fetch_add(1, cuda::memory_order_relaxed);
    }
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBinHeader() {
  std::printf("%-12s", "bin");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" [%3u-%3u]", b * INTENSITIES_PER_BIN, (b + 1) * INTENSITIES_PER_BIN - 1);
  }
  std::printf("\n");
}

static void printBins(const char* label, const unsigned int* bins) {
  std::printf("%-12s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
  }
  std::printf("\n");
}

static bool sameBins(const unsigned int* bins, const unsigned int* expected) {
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

static unsigned int contiguousPixel(unsigned int thread, unsigned int c) {
  return thread * COARSE_FACTOR + c;
}

static unsigned int interleavedPixel(unsigned int thread, unsigned int c) {
  return c * THREADS_PER_BLOCK + thread;
}

static void printPixelsReadByBlock0(const char* name,
                                    unsigned int (*pixelOf)(unsigned int, unsigned int)) {
  std::printf("%s: pixels read by threads 0..%u of block 0\n", name, THREADS_PER_BLOCK - 1);
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    std::printf("  c = %u:", c);
    for (unsigned int thread = 0; thread < THREADS_PER_BLOCK; ++thread) {
      std::printf(" %3u", pixelOf(thread, c));
    }
    std::printf("\n");
  }
}

template <class Kernel>
static void runKernel(Kernel kernel, const unsigned char* image_d, unsigned int* bins) {
  unsigned int* bins_d = nullptr;
  CHECK(cudaMalloc(&bins_d, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMemset(bins_d, 0, NUM_BINS * sizeof(unsigned int)));

  const unsigned int numBlocks = cdiv(NUM_PIXELS, COARSE_FACTOR * THREADS_PER_BLOCK);
  launch(kernel, dim3(numBlocks), dim3(THREADS_PER_BLOCK), image_d, bins_d, WIDTH, HEIGHT);
  CHECK(cudaGetLastError());

  CHECK(cudaMemcpy(bins, bins_d, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(bins_d));
}

int main() {
  unsigned char* image_d = nullptr;
  CHECK(cudaMalloc(&image_d, NUM_PIXELS));
  CHECK(cudaMemcpy(image_d, &TREE_IMAGE[0][0], NUM_PIXELS, cudaMemcpyHostToDevice));

  unsigned int contiguous[NUM_BINS];
  unsigned int interleaved[NUM_BINS];
  runKernel(histogram_coarse_contig, image_d, contiguous);
  runKernel(histogram_coarse_interl, image_d, interleaved);
  CHECK(cudaFree(image_d));

  unsigned int expected[NUM_BINS] = {0, 0, 0, 0};
  histogram_sequential(&TREE_IMAGE[0][0], expected, WIDTH, HEIGHT);

  printImage();
  std::printf("\n%u blocks of %u threads, COARSE_FACTOR = %u pixels per thread\n",
              cdiv(NUM_PIXELS, COARSE_FACTOR * THREADS_PER_BLOCK),
              THREADS_PER_BLOCK,
              COARSE_FACTOR);
  printPixelsReadByBlock0("contiguous", contiguousPixel);
  printPixelsReadByBlock0("interleaved", interleavedPixel);
  std::printf("\n");
  printBinHeader();
  printBins("contiguous", contiguous);
  printBins("interleaved", interleaved);
  printBins("sequential", expected);

  const bool ok = sameBins(contiguous, expected) && sameBins(interleaved, expected);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_coarsening.cu -o 05_coarsening && ./05_coarsening

**What to look at in the output:** In iteration c = 0 the contiguous threads read pixels 0, 4, 8, ... and the interleaved threads read 0, 1, 2, ...; both kernels give the same bins.

## Step 6 of 7: `06_most_recent_bin.cu`

`histogram_thread_run`: each thread keeps the most recent bin in a register (`bin_r`) and issues one shared-memory atomic per run of equal bins. It prints the bins each thread sees (a column of the image) and counts 30 atomics instead of 64. (slide: *Code: privatizing the most recent bin*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 06_most_recent_bin.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_PIXELS = WIDTH * HEIGHT;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;
constexpr unsigned int THREADS_PER_BLOCK = 8;
constexpr unsigned int COARSE_FACTOR = 4;
constexpr unsigned int NUM_BLOCKS = NUM_PIXELS / (COARSE_FACTOR * THREADS_PER_BLOCK);

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_thread_run(const unsigned char* image,
                                     unsigned int* bins,
                                     unsigned int width,
                                     unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  if (segment + threadIdx.x < width * height) {
    unsigned int b = image[segment + threadIdx.x] / INTENSITIES_PER_BIN;
    unsigned int bin_r = 1;
    for (unsigned int c = 1; c < COARSE_FACTOR; ++c) {
      unsigned int i = segment + c * blockDim.x + threadIdx.x;
      if (i < width * height) {
        unsigned int bNext = image[i] / INTENSITIES_PER_BIN;
        if (bNext == b) {
          ++bin_r;
        } else {
          cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
          bins_s_ref.fetch_add(bin_r, cuda::memory_order_relaxed);
          b = bNext;
          bin_r = 1;
        }
      }
    }
    cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
    bins_s_ref.fetch_add(bin_r, cuda::memory_order_relaxed);
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBinHeader() {
  std::printf("%-12s", "bin");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" [%3u-%3u]", b * INTENSITIES_PER_BIN, (b + 1) * INTENSITIES_PER_BIN - 1);
  }
  std::printf("\n");
}

static void printBins(const char* label, const unsigned int* bins) {
  std::printf("%-12s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
  }
  std::printf("\n");
}

static bool sameBins(const unsigned int* bins, const unsigned int* expected) {
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

static unsigned int binOfPixel(unsigned int pixel) {
  return (&TREE_IMAGE[0][0])[pixel] / INTENSITIES_PER_BIN;
}

static unsigned int printRunsOfThread(unsigned int block, unsigned int thread) {
  unsigned int segment = COARSE_FACTOR * block * THREADS_PER_BLOCK;
  unsigned int atomics = 1;
  std::printf("  block %u thread %u reads bins", block, thread);
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int pixel = segment + c * THREADS_PER_BLOCK + thread;
    std::printf(" %u", binOfPixel(pixel));
    if (c > 0 && binOfPixel(pixel) != binOfPixel(pixel - THREADS_PER_BLOCK)) {
      ++atomics;
    }
  }
  std::printf("  -> %u shared atomic%s\n", atomics, atomics == 1 ? "" : "s");
  return atomics;
}

int main() {
  unsigned char* image_d = nullptr;
  unsigned int* bins_d = nullptr;
  CHECK(cudaMalloc(&image_d, NUM_PIXELS));
  CHECK(cudaMalloc(&bins_d, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMemcpy(image_d, &TREE_IMAGE[0][0], NUM_PIXELS, cudaMemcpyHostToDevice));
  CHECK(cudaMemset(bins_d, 0, NUM_BINS * sizeof(unsigned int)));

  launch(histogram_thread_run,
         dim3(NUM_BLOCKS),
         dim3(THREADS_PER_BLOCK),
         image_d,
         bins_d,
         WIDTH,
         HEIGHT);
  CHECK(cudaGetLastError());

  unsigned int bins[NUM_BINS];
  CHECK(cudaMemcpy(bins, bins_d, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(image_d));
  CHECK(cudaFree(bins_d));

  unsigned int expected[NUM_BINS] = {0, 0, 0, 0};
  histogram_sequential(&TREE_IMAGE[0][0], expected, WIDTH, HEIGHT);

  printImage();
  std::printf(
      "\n%u blocks of %u threads, COARSE_FACTOR = %u, interleaved: thread t walks down column t\n",
      NUM_BLOCKS,
      THREADS_PER_BLOCK,
      COARSE_FACTOR);
  unsigned int totalAtomics = 0;
  for (unsigned int block = 0; block < NUM_BLOCKS; ++block) {
    for (unsigned int thread = 0; thread < THREADS_PER_BLOCK; ++thread) {
      totalAtomics += printRunsOfThread(block, thread);
    }
  }
  std::printf("shared-memory atomics: %u (one per run) instead of %u (one per pixel)\n\n",
              totalAtomics,
              NUM_PIXELS);

  printBinHeader();
  printBins("bins", bins);
  printBins("sequential", expected);

  const bool ok = sameBins(bins, expected);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 06_most_recent_bin.cu -o 06_most_recent_bin && ./06_most_recent_bin

**What to look at in the output:** Threads whose column is all sky need a single atomic; the total line shows 30 shared-memory atomics instead of 64.

## Step 7 of 7: `07_thread_private_bins.cu`

`histogram_thread_private`: each thread counts its 4 interleaved pixels in a private array `bins_t[NUM_BINS]` with plain `+=` (the unrolled compare-and-add keeps it in registers), then adds each nonzero entry to `bins_s` with one block-scope atomic. It prints every thread's `bins_t` and counts the shared-memory atomics. (slide: *Code: thread-level privatization*)

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 07_thread_private_bins.cu
#include "cuda_compat.h"

#include <cstdio>

constexpr unsigned int WIDTH = 8;
constexpr unsigned int HEIGHT = 8;
constexpr unsigned int NUM_PIXELS = WIDTH * HEIGHT;
constexpr unsigned int NUM_INTENSITIES = 256;
constexpr unsigned int NUM_BINS = 4;
constexpr unsigned int INTENSITIES_PER_BIN = NUM_INTENSITIES / NUM_BINS;
constexpr unsigned int THREADS_PER_BLOCK = 8;
constexpr unsigned int COARSE_FACTOR = 4;
constexpr unsigned int NUM_BLOCKS = NUM_PIXELS / (COARSE_FACTOR * THREADS_PER_BLOCK);

const unsigned char TREE_IMAGE[HEIGHT][WIDTH] = {
    {231, 236, 242, 247, 250, 245, 239, 233},
    {228, 234, 240, 96, 102, 238, 232, 226},
    {225, 230, 88, 110, 115, 92, 229, 224},
    {222, 227, 84, 105, 118, 90, 226, 221},
    {220, 224, 80, 35, 42, 86, 223, 219},
    {218, 222, 216, 33, 40, 215, 220, 217},
    {150, 156, 162, 30, 38, 165, 158, 152},
    {145, 160, 170, 175, 168, 172, 155, 148},
};

void histogram_sequential(const unsigned char* image,
                          unsigned int* bins,
                          unsigned int width,
                          unsigned int height) {
  for (unsigned int i = 0; i < width * height; ++i) {
    unsigned char b = image[i] / INTENSITIES_PER_BIN;
    ++bins[b];
  }
}

__global__ void histogram_thread_private(const unsigned char* image,
                                         unsigned int* bins,
                                         unsigned int width,
                                         unsigned int height) {
  __shared__ unsigned int bins_s[NUM_BINS];
  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    bins_s[b] = 0;
  }
  __syncthreads();

  unsigned int bins_t[NUM_BINS] = {};
  unsigned int segment = COARSE_FACTOR * blockIdx.x * blockDim.x;
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int i = segment + c * blockDim.x + threadIdx.x;
    if (i < width * height) {
      unsigned int bin = image[i] / INTENSITIES_PER_BIN;
#pragma unroll
      for (unsigned int b = 0; b < NUM_BINS; ++b) {
        bins_t[b] += (bin == b);
      }
    }
  }
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins_t[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_block> bins_s_ref(bins_s[b]);
      bins_s_ref.fetch_add(bins_t[b], cuda::memory_order_relaxed);
    }
  }
  __syncthreads();

  for (unsigned int b = threadIdx.x; b < NUM_BINS; b += blockDim.x) {
    if (bins_s[b] > 0) {
      cuda::atomic_ref<unsigned int, cuda::thread_scope_device> bins_ref(bins[b]);
      bins_ref.fetch_add(bins_s[b], cuda::memory_order_relaxed);
    }
  }
}

static void printImage() {
  std::printf("image (%u x %u intensities):\n", WIDTH, HEIGHT);
  for (unsigned int row = 0; row < HEIGHT; ++row) {
    for (unsigned int col = 0; col < WIDTH; ++col) {
      std::printf(" %3u", TREE_IMAGE[row][col]);
    }
    std::printf("\n");
  }
}

static void printBinHeader() {
  std::printf("%-12s", "bin");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" [%3u-%3u]", b * INTENSITIES_PER_BIN, (b + 1) * INTENSITIES_PER_BIN - 1);
  }
  std::printf("\n");
}

static void printBins(const char* label, const unsigned int* bins) {
  std::printf("%-12s", label);
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %9u", bins[b]);
  }
  std::printf("\n");
}

static bool sameBins(const unsigned int* bins, const unsigned int* expected) {
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    if (bins[b] != expected[b]) {
      return false;
    }
  }
  return true;
}

static unsigned int binOfPixel(unsigned int pixel) {
  return (&TREE_IMAGE[0][0])[pixel] / INTENSITIES_PER_BIN;
}

static unsigned int printPrivateBinsOfThread(unsigned int block, unsigned int thread) {
  unsigned int segment = COARSE_FACTOR * block * THREADS_PER_BLOCK;
  unsigned int bins_t[NUM_BINS] = {};
  std::printf("  block %u thread %u reads bins", block, thread);
  for (unsigned int c = 0; c < COARSE_FACTOR; ++c) {
    unsigned int pixel = segment + c * THREADS_PER_BLOCK + thread;
    std::printf(" %u", binOfPixel(pixel));
    ++bins_t[binOfPixel(pixel)];
  }
  unsigned int atomics = 0;
  std::printf("  bins_t =");
  for (unsigned int b = 0; b < NUM_BINS; ++b) {
    std::printf(" %u", bins_t[b]);
    if (bins_t[b] > 0) {
      ++atomics;
    }
  }
  std::printf("  -> %u shared atomic%s\n", atomics, atomics == 1 ? "" : "s");
  return atomics;
}

int main() {
  unsigned char* image_d = nullptr;
  unsigned int* bins_d = nullptr;
  CHECK(cudaMalloc(&image_d, NUM_PIXELS));
  CHECK(cudaMalloc(&bins_d, NUM_BINS * sizeof(unsigned int)));
  CHECK(cudaMemcpy(image_d, &TREE_IMAGE[0][0], NUM_PIXELS, cudaMemcpyHostToDevice));
  CHECK(cudaMemset(bins_d, 0, NUM_BINS * sizeof(unsigned int)));

  launch(histogram_thread_private,
         dim3(NUM_BLOCKS),
         dim3(THREADS_PER_BLOCK),
         image_d,
         bins_d,
         WIDTH,
         HEIGHT);
  CHECK(cudaGetLastError());

  unsigned int bins[NUM_BINS];
  CHECK(cudaMemcpy(bins, bins_d, NUM_BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(image_d));
  CHECK(cudaFree(bins_d));

  unsigned int expected[NUM_BINS] = {0, 0, 0, 0};
  histogram_sequential(&TREE_IMAGE[0][0], expected, WIDTH, HEIGHT);

  printImage();
  std::printf(
      "\n%u blocks of %u threads, COARSE_FACTOR = %u, interleaved: thread t walks down column t\n",
      NUM_BLOCKS,
      THREADS_PER_BLOCK,
      COARSE_FACTOR);
  unsigned int totalAtomics = 0;
  for (unsigned int block = 0; block < NUM_BLOCKS; ++block) {
    for (unsigned int thread = 0; thread < THREADS_PER_BLOCK; ++thread) {
      totalAtomics += printPrivateBinsOfThread(block, thread);
    }
  }
  std::printf(
      "shared-memory atomics: %u (one per nonzero bins_t entry) instead of %u (one per pixel)\n\n",
      totalAtomics,
      NUM_PIXELS);

  printBinHeader();
  printBins("bins", bins);
  printBins("sequential", expected);

  const bool ok = sameBins(bins, expected);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 07_thread_private_bins.cu -o 07_thread_private_bins && ./07_thread_private_bins

**What to look at in the output:** Each thread's `bins_t` adds up to its 4 pixels and needs at most 4 atomics (one per nonzero bin); the total line shows 30 shared-memory atomics instead of 64, and the counting loop has none.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/07_histogram/README.md`.

**Next:** open `colab_2_full_programs.ipynb` in this same folder.